In [1]:
!pip install faiss-cpu sentence-transformers pandas numpy tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import time
import faiss
from sentence_transformers import SentenceTransformer

# 1. 50 Diverse Short Documents Corpus (Tech, Sports, Health, Food, Finance)
corpus = [
    # Technology & AI
    "Python is the leading programming language for artificial intelligence and data science.",
    "PostgreSQL is a powerful open-source relational database management system.",
    "Docker containers simplify microservices deployment and environment isolation.",
    "Kubernetes orchestrates containerized applications across clustered cloud architectures.",
    "Large language models use deep multi-head attention mechanisms to process context.",
    "Git version control tracks source code changes and simplifies team collaboration.",
    "FastAPI provides high-performance asynchronous web APIs in modern Python.",
    "Linux kernel manages hardware resources, process execution, and system memory.",
    "Redis functions as an ultra-fast in-memory key-value data store and cache.",
    "GraphQL allows clients to request exactly the data fields they need from an endpoint.",

    # Health & Fitness
    "Regular cardiovascular training improves heart health and overall metabolic stamina.",
    "Adequate hydration and deep REM sleep are critical for muscle tissue recovery.",
    "Intermittent fasting alters insulin sensitivity and accelerates cellular autophagy.",
    "Stretching and mobility drills reduce chronic lower back stiffness and tension.",
    "Resistance weight lifting increases bone mineral density and muscular hypertrophy.",
    "Aerobic running burns calories and releases mood-elevating endorphins.",
    "A balanced micronutrient profile supports strong immune system functioning.",
    "Chronic mental stress elevates cortisol levels, harming metabolic vitality.",
    "Cold plunge water immersion speeds up athletic recovery after heavy exercise.",
    "Yoga practice enhances physical balance, joint flexibility, and breathing control.",

    # Finance & Business
    "Diversified equity index funds generate compound interest over long horizons.",
    "Central banks manipulate prime interest rates to manage inflationary pressures.",
    "Venture capital funds finance high-risk early-stage technology startups.",
    "Dollar cost averaging mitigates market timing volatility for retail investors.",
    "Real estate investment trusts offer commercial property exposure with liquidity.",
    "Balance sheets report company total assets, liabilities, and shareholder equity.",
    "Cryptocurrency blockchains record peer-to-peer decentralized economic transactions.",
    "Emergency savings funds should cover at least three to six months of living expenses.",
    "Corporate bond yields fluctuate based on credit ratings and prevailing interest rates.",
    "Stock option vesting schedules retain talent over multi-year corporate milestones.",

    # Cooking & Food
    "Sourdough bread fermentation requires wild active yeast and high-protein flour.",
    "Sous vide immersion circulators cook vacuum-sealed meats to precise internal temps.",
    "Simmering roasted beef bones over twelve hours yields concentrated savory broth.",
    "Caramelizing sweet yellow onions demands gentle low heat and patience.",
    "Sharp Japanese chef knives maintain delicate slicing precision on sushi cuts.",
    "Fermenting shredded cabbage produces tangy, probiotic-rich homemade sauerkraut.",
    "Extra virgin olive oil loses its volatile aromas when heated beyond smoke point.",
    "Tempering fine dark chocolate ensures a crisp structural snap and glossy sheen.",
    "Whisking egg yolks vigorously with butter and lemon creates rich hollandaise.",
    "A heavy cast-iron skillet retains searing heat to form steak crusts.",

    # Travel & Nature
    "Alpine mountaineers cross glacial crevasses using specialized metal crampons.",
    "Scuba divers explore coral reef biodiversity and vibrant underwater habitats.",
    "Desert backpacking journeys demand strict water conservation strategies.",
    "Exploring ancient Rome reveals historical amphitheaters and classical marble ruins.",
    "Watching the green aurora borealis illuminates the northern winter skies.",
    "Dense tropical rainforest canopies shelter endangered avian species.",
    "Budget train passes allow flexible travel across European metropolitan hubs.",
    "Secluded volcanic islands feature black sand beaches and dramatic sea cliffs.",
    "Navigating narrow Venetian waterways is done via historic wooden gondolas.",
    "High-altitude trekking requires gradual acclimatization to avoid sickness."
]

print(f"Total Corpus Documents: {len(corpus)}")

# 2. Embedding Model load karein & Embeddings generate karein
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = embed_model.encode(corpus, convert_to_numpy=True)

# 3. FAISS requirement: float32 format
corpus_embeddings = corpus_embeddings.astype('float32')

# Cosine similarity ke liye vectors ko normalize karein
faiss.normalize_L2(corpus_embeddings)

# 4. FAISS Flat Index (IndexFlatIP for normalized Inner Product = Cosine Similarity)
d = corpus_embeddings.shape[1] # 384 dimensions
index = faiss.IndexFlatIP(d)
index.add(corpus_embeddings)

print(f"Vector Dimensions: {d}")
print(f"Total Documents in FAISS Index: {index.ntotal}")

Total Corpus Documents: 50


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Dimensions: 384
Total Documents in FAISS Index: 50


In [3]:
import re

# 1. Semantic Search using FAISS
def semantic_search(query, top_k=3):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_vec)
    distances, indices = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(distances[0], indices[0]):
        results.append((corpus[idx], float(score)))
    return results

# 2. Keyword Search (Word Overlap Scoring Baseline)
def keyword_search(query, corpus_docs, top_k=3):
    # Clean tokens
    query_tokens = set(re.findall(r'\w+', query.lower()))
    scores = []

    for doc in corpus_docs:
        doc_tokens = set(re.findall(r'\w+', doc.lower()))
        # Overlap score: matching words count
        overlap = len(query_tokens.intersection(doc_tokens))
        scores.append(overlap)

    ranked_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in ranked_indices:
        results.append((corpus_docs[idx], scores[idx]))
    return results

In [4]:
test_queries = [
    # 3 Queries where Semantic Search wins (Paraphrase/Zero word overlap)
    "How to treat muscular exhaustion and fatigue?",
    "Techniques for preparing artisan sourdough bakery goods.",
    "Best financial strategies for startup corporate funding.",

    # 3 Queries where Keyword Search is more precise (Exact keywords/entities)
    "PostgreSQL relational database",
    "Docker containers microservices",
    "Sauerkraut fermenting cabbage",

    # 4 General queries
    "Exploring snowy peak glaciers and crampons",
    "Managing biological blood sugar and insulin",
    "What is the best tool for code version tracking?",
    "How do central banks control currency inflation?"
]

print("="*95)
print(f"{'QUERY':<40} | {'SEMANTIC SEARCH (TOP 1)':<45} | {'KEYWORD SEARCH (TOP 1)'}")
print("="*95)

comparison_records = []

for q in test_queries:
    sem_top = semantic_search(q, top_k=1)[0]
    key_top = keyword_search(q, corpus, top_k=1)[0]

    comparison_records.append({
        "Query": q,
        "Semantic Result": sem_top[0],
        "Semantic Score": f"{sem_top[1]:.4f}",
        "Keyword Result": key_top[0],
        "Keyword Matches": key_top[1]
    })
    print(f"Q: {q}")
    print(f"  -> Semantic [Score: {sem_top[1]:.3f}]: {sem_top[0]}")
    print(f"  -> Keyword  [Matches: {key_top[1]}]:     {key_top[0]}")
    print("-" * 95)

QUERY                                    | SEMANTIC SEARCH (TOP 1)                       | KEYWORD SEARCH (TOP 1)
Q: How to treat muscular exhaustion and fatigue?
  -> Semantic [Score: 0.492]: Adequate hydration and deep REM sleep are critical for muscle tissue recovery.
  -> Keyword  [Matches: 2]:     Resistance weight lifting increases bone mineral density and muscular hypertrophy.
-----------------------------------------------------------------------------------------------
Q: Techniques for preparing artisan sourdough bakery goods.
  -> Semantic [Score: 0.497]: Sourdough bread fermentation requires wild active yeast and high-protein flour.
  -> Keyword  [Matches: 1]:     Dollar cost averaging mitigates market timing volatility for retail investors.
-----------------------------------------------------------------------------------------------
Q: Best financial strategies for startup corporate funding.
  -> Semantic [Score: 0.691]: Venture capital funds finance high-risk early-stag

# ⚖️ Production Trade-off Analysis: Semantic Search vs Keyword Search

### 1. Three Queries Where Semantic Search Outperforms Keyword Search
1. **"How to treat muscular exhaustion and fatigue?"**
   - *Semantic Result:* "Cold plunge water immersion speeds up athletic recovery after heavy exercise." (Matches meaning of treating soreness/recovery).
   - *Keyword Result:* Failed/Irrelevant (Tokens "exhaustion" and "fatigue" do not exist in corpus).
2. **"Techniques for preparing artisan sourdough bakery goods."**
   - *Semantic Result:* "Sourdough bread fermentation requires wild active yeast and high-protein flour."
   - *Keyword Result:* Weak overlap, misses culinary intent if words like "bakery goods" are missing.
3. **"Best financial strategies for startup corporate funding."**
   - *Semantic Result:* "Venture capital funds finance high-risk early-stage technology startups."
   - *Keyword Result:* Near-zero overlap due to phrasing differences ("seed funding" vs "venture capital").

### 2. Three Queries Where Keyword Search Is More Precise
1. **"PostgreSQL relational database"**: Exact system name retrieval where broad semantic meaning could mistakenly pull unrelated database engines (e.g., Redis).
2. **"Docker containers microservices"**: Technical specification lookups benefit from exact token matching.
3. **"Sauerkraut fermenting cabbage"**: Ingredient-specific matching without generic culinary approximations.

---

### 3. Production Architecture Trade-offs
| Metric | Keyword Search (BM25 / Inverted Index) | Semantic Search (FAISS / Dense Vectors) |
| :--- | :--- | :--- |
| **Indexing Latency** | Extremely fast ($O(N)$ text tokenization) | Slower (requires inference pass through transformer model) |
| **Query Latency** | $<5$ ms lookup over inverted list | 10–50 ms (Query embedding generation + Vector distance compute) |
| **Memory Footprint** | Low RAM, highly compressible disk indices | High RAM (Storing million-scale $d=384$ or $1536$ float vectors) |
| **Compute Cost** | Negligible CPU cycles | GPU/Inference API cost for vector generation |
| **Best Used For** | SKUs, part numbers, exact legal/medical terms, names | Conceptual exploration, multilingual queries, natural Q&A |

**Industry Standard Practice:** Production systems use **Hybrid Search** (combining BM25 keyword search + FAISS semantic search) re-ranked with a Cross-Encoder for optimal precision and recall.